# C-33 · Source citations and lineage
**Requirement:** each answer lists the documents, rows and model version it used.

**Where the information comes from.** AIDB's debug stream (`agent_converse(..., debug => true)`) reports every tool call, its arguments and its result. `lab.ask` keeps, for each tool call, the **tool, the arguments and the ids of the rows returned**, and stores them with the model name and version in `guard.events.sources`. A citation is then a lookup of those ids.

In [1]:
import lab
lab.install()   # creates the guard schema, sample PII, tools, agents and purposes (safe to run again)

guard schema, sample contacts and personas: ready
tools: g_list_tickets, g_list_contacts, g_search_articles, g_assign_ticket, g_list_tickets_rls, g_ticket_notes
purposes: guard_readonly (guard_ro), guard_triage (guard_rw)
agents: guard_reader, guard_triager, guard_persona
intent examples registered for the agents


## 1 · Ask, and keep the lineage

In [2]:
res = lab.ask("guard_reader", "Which tickets are critical?")

[ALLOWED] guard_reader <- 'Which tickets are critical?'
  intent : similarity 0.858
  answer : Summarized Status Report:
- The task involved determining which tickets are critical.
- Initial query and tool calls returned the following critical tickets:
  - ID: 1001, Assignee: maria, Customer: Aurora Bank, Subject: WAL volume at 97% on primary, writes failing, Status: open
  - ID: 1002, Assignee: jonas, Customer: Delta Health, Subject: Need PITR to 09:15 after bad deployment, Status: in_progress

The tool calls for retrieving critical tickets have returned the same results repeatedly without progressing beyond this information. No final answer has been produced due to continuous identica
  source : g_list_tickets({"assignee": "any", "priority": "critical", "status": "any"}) -> [{"id": 1001}, {"id": 1002}]
  source : g_list_tickets({"assignee": "any", "priority": "critical", "status": "any"}) -> [{"id": 1001}, {"id": 1002}]
  source : g_list_tickets({"assignee": "any", "priority": "criti

In [3]:
res2 = lab.ask("guard_reader", "How do I fix a full WAL volume?")

[ALLOWED] guard_reader <- 'How do I fix a full WAL volume?'
  intent : similarity 0.888
  answer : When the WAL volume fills up Postgres stops accepting writes and may shut down. First find out what is holding WAL back: an inactive replication slot is the usual cause. Check pg_replication_slots for slots with active = false and a large restart_lsn lag. If the consumer is gone for good, drop the slot with pg_drop_replication_slot(). Then run a CHECKPOINT and confirm that old segments in pg_wal are no longer referenced by pg_xlog.fsnodemap.
  source : g_searchArticles({"keyword": "WAL"}) -> []
  source : g_search_articles({"keyword": "WAL"}) -> [{"id": 1, "title": "Disk full on the WAL volume"}, {"id": 2, "title": "Restoring a database to a point in time (PITR)"}, {"id": 5, "title": "Failover runbook for a streaming replica"}]
  source : g_search_articles({"keyword": "WAL"}) -> [{"id": 1, "title": "Disk full on the WAL volume"}, {"id": 2, "title": "Restoring a database to a point in time

## 2 · Turn the lineage into citations
Ids are resolved against the source tables, so the citation names the actual ticket subject or runbook title.

In [4]:
import json
LOOKUP = {  # tool -> how to describe a row it returned
    "g_list_tickets":    ("ticket",  "SELECT id, subject FROM support.tickets WHERE id = ANY(%s)"),
    "g_search_articles": ("runbook", "SELECT id, title  FROM support.articles WHERE id = ANY(%s)"),
    "g_list_contacts":   ("contact", "SELECT id, name   FROM support.contacts WHERE id = ANY(%s)"),
}

def cite(res):
    lines = []
    for s in res["sources"]:
        if s["tool"] not in LOOKUP or not s["rows"]:
            continue
        kind, q = LOOKUP[s["tool"]]
        ids = [r["id"] for r in s["rows"]]
        for i, label in lab.sql(q, (ids,), quiet=True)[0]:
            lines.append(f"  [{kind} #{i}] {label}   (via {s['tool']})")
    model = lab.sql("SELECT model, model_version FROM guard.events WHERE id = %s", (res["event_id"],), role="demo_governor", quiet=True)[0][0]
    lines.append(f"  model: {model[0]} -> {model[1]}")
    return "Sources\n" + "\n".join(sorted(set(lines)))

print(cite(res)); print(); print(cite(res2))

Sources
  [ticket #1001] WAL volume at 97% on primary, writes failing   (via g_list_tickets)
  [ticket #1002] Need PITR to 09:15 after bad deployment   (via g_list_tickets)
  model: desk_llm -> completions:qwen2.5:7b

Sources
  [runbook #1] Disk full on the WAL volume   (via g_search_articles)
  [runbook #2] Restoring a database to a point in time (PITR)   (via g_search_articles)
  [runbook #5] Failover runbook for a streaming replica   (via g_search_articles)
  model: desk_llm -> completions:qwen2.5:7b


## 3 · The lineage is stored with the event
So it can be audited later, and queried.

In [5]:
lab.sql("""SELECT id, agent, model_version, jsonb_pretty(sources) AS sources
          FROM guard.events WHERE sources IS NOT NULL AND jsonb_array_length(sources) > 0
          ORDER BY id DESC LIMIT 1""", role="demo_governor", width=400)

id,agent,model_version,sources
15,guard_reader,completions:qwen2.5:7b,"[ { ""rows"": [ ], ""tool"": ""g_searchArticles"", ""error"": ""tool 'g_searchArticles' is not available to this agent"", ""arguments"": { ""keyword"": ""WAL"" } }, { ""rows"": [ { ""id"": 1, ""title"": ""Disk full on the WAL volume"" }, { ""id"": 2, …"


## 4 · Reverse lineage: who was shown ticket 1001?

In [6]:
lab.sql("""SELECT e.id, e.at::time(0) AS at, e.actor, e.agent, e.model_version, s->>'tool' AS tool
          FROM guard.events e, jsonb_array_elements(e.sources) s, jsonb_array_elements(s->'rows') r
          WHERE (r->>'id')::int = 1001 AND s->>'tool' = 'g_list_tickets'
          GROUP BY 1, 2, 3, 4, 5, 6 ORDER BY e.id DESC LIMIT 5""", role="demo_governor")

id,at,actor,agent,model_version,tool
14,12:02:18,demo_user,guard_reader,completions:qwen2.5:7b,g_list_tickets
4,12:00:54,demo_user,guard_reader,completions:qwen2.5:7b,g_list_tickets
2,12:00:39,demo_user,guard_reader,completions:qwen2.5:7b,g_list_tickets


## What to look for
* The citations come from what the **tools actually returned**, not from what the model says it used, so a model cannot cite a source it never saw.
* `model_version` is the provider and model tag at the time of the request (for example `completions:qwen2.5:7b`). If you swap the model, old rows keep the old value.
* Lineage is by row id. For the runbook search (a vector search over a knowledge base), a `search_runbooks`-style tool would return the article title and excerpt; point `LOOKUP` at it the same way.
* A weak local model sometimes calls a tool several times or ignores the result in its wording. The lineage records what really happened either way.